# 第 25 章 · 让 GPU 别等：数据管道与同步点

本章回答：怎么量出"GPU 有多少时间在等数据"？从读硬盘到进显存，每一段怎么提速？训练循环里藏着哪些同步点，怎么一次性把它们找出来？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/25-input-sync.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch13「优化数据输入流水线」（`ch13.md` 第 1552–1561 行）、「重叠通信和计算」中的数据预取例子（第 1021–1097 行）；官方文档 CUDA semantics「Use pinned memory buffers」、`torch.cuda.set_sync_debug_mode`（〔在线〕）；本讲义 [第 5 章](05-cuda-semantics.md)、[第 8 章](08-dataloader.md)

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 等数据占比 | 18 ÷ 48 | `data_wait_share(18, 30)` | 37.5% |
| 第三步 GPU 上累计 | 最后只取一次 | `accumulate_on_device(...)` | 2.5 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `accumulate_on_device`**（来自 `ch25_input_sync.py`）：指标在张量上累加，只在最后 .item() 一次（同步一次）。

In [ ]:
def accumulate_on_device(losses):
    """指标在张量上累加，只在最后 .item() 一次（同步一次）。"""
    running = torch.zeros(())
    for loss in losses:
        running += loss.detach()
    return (running / len(losses)).item()

**函数 `data_wait_share`**（来自 `ch25_input_sync.py`）：下面要用到的小工具。

In [ ]:
def data_wait_share(t_data_ms: float, t_compute_ms: float) -> float:
    return t_data_ms / (t_data_ms + t_compute_ms)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 等数据 18 ms、计算 30 ms → GPU 等待占比", f"{data_wait_share(18, 30):.1%}")

In [ ]:
print("\n第三步 在张量上累加 4 个 loss，最后取一次：", accumulate_on_device([torch.tensor(v) for v in (1.0, 2.0, 3.0, 4.0)]))

## 3. 代码里才看得到的

1. **`detach()` 别忘了**：不然累加会把每一步的账本都连起来，越占越多。

## 4. 自己改一改

- 在有 CUDA 的机器上加 `torch.cuda.set_sync_debug_mode("warn")`，在循环里调 `.item()` 看警告。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 先拆两段计时，数据不慢别瞎改；
> 读取、处理、拼盘、拷贝四段路，能离线就离线，能并行就并行，拷贝用 pin 加 non_blocking；
> 同步点藏在日志和指标里，sync_debug_mode 帮你找；第二条 stream 提前拷下一批。